# Data Loading and Preparation

## Basic Imports

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Modelling
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score, make_scorer
import joblib # For persistence

## Load Processed Feature Dataset for Modelling

In [ ]:
df_model = pd.read_csv("../data/processed/df_features.csv")
print("Loaded dataset shape:", df_model.shape)
display(df_model.head())

## Define Features

In [ ]:
features = [
    "log_dbytes",
    "log_min_flow_bytes",
    "log_byte_prod",
    "Dload",
    "load_skew",
    "log_dmeansz",
    "mean_pkt_sz_ratio",
    "sttl",
    "dttl",
    "ttl_diff",
    "ct_state_ttl",
    "ackdat",
    "synack"
] # Selected features

## Train/Test Split

In [ ]:
df_train, _ = train_test_split(
    df_model, test_size=0.2, stratify=df_model["Label"], random_state=42
)

X_train = df_train[features]
y_train = df_train["Label"]

attack_fraction = y_train.sum()/len(y_train)
print("Contamination fraction:", attack_fraction)

# Model Fit

## Isolation Forest Model

### Initialise Model

In [ ]:
model = Pipeline([
    ("scaler", RobustScaler()),
    ("iforest", IsolationForest(
        n_estimators=500,
        max_samples=250,
        contamination=0.075,
        random_state=42,
        n_jobs=-1
    ))
])

### Fit Model

In [ ]:
print("Fitting Isolation Forest...")
model.fit(X_train)
print("Model fitting complete.")

### Hyperparameter Experiments (Optional)

This section is only needed if you want to explore different hyperparameter configurations. The final model uses the configuration above.

In [ ]:
# Custom scorer: attack f1
def attack_f1(y_true, y_pred):
    y_pred_mapped = np.where(y_pred == -1, 1, 0)
    return f1_score(y_true, y_pred_mapped)
f1_attack_scorer = make_scorer(attack_f1)

search_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("iforest", IsolationForest(random_state=42, n_jobs=-1))
])

# Hyperparameter grid
param_dist = {
    "iforest__n_estimators": [500, 1000, 1500],
    "iforest__max_samples": [150, 200, 250],
    "iforest__contamination": [0.06, 0.075, 0.09]
}

# Randomised search
search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_dist,
    n_iter=27,
    cv=4,
    scoring=f1_attack_scorer,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# Fit search
print("Running hyperparameter search...")
search.fit(X_train, y_train)
print("Hyperparameter search complete.")

# Inspect results
print("Best hyperparameters:", search.best_params_)
print("Best F1 (attack) score:", search.best_score_)
results_df = pd.DataFrame(search.cv_results_)

# Model Persistence

In [ ]:
joblib.dump(model, "../models/isolation_forest_pipeline.pkl")
print("Model saved.")